# Machine Learning Zoomcamp 2026 - Homework 2
## Module: Machine Learning for Regression

In this homework, we build a linear regression model to predict car fuel efficiency (`fuel_efficiency_mpg`) using vehicle features.

In [ ]:
import numpy as np
import pandas as pd

# Dataset URL
DATA_URL = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv'
df_raw = pd.read_csv(DATA_URL)

FEATURES = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year']
TARGET = 'fuel_efficiency_mpg'

df = df_raw[FEATURES + [TARGET]].copy()
df.head()

### Question 1: Missing Values
There's one column with missing values. What is it?

In [ ]:
missing = df[FEATURES].isnull().sum()
print(missing)
q1_ans = missing[missing > 0].index[0]
print(f'\nAnswer Q1: {q1_ans}')

### Question 2: Median for horsepower
What's the median (50% percentile) for variable `'horsepower'`?

In [ ]:
median_hp = df['horsepower'].median()
print(f'Horsepower Median: {median_hp}')
print(f'Answer Q2: {int(median_hp)}')

### Helper Functions: Split, Training, and RMSE

In [ ]:
def split_data(df, seed=42):
    n = len(df)
    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test

    idx = np.arange(n)
    np.random.RandomState(seed).shuffle(idx)

    train = df.iloc[idx[:n_train]].copy().reset_index(drop=True)
    val = df.iloc[idx[n_train:n_train + n_val]].copy().reset_index(drop=True)
    test = df.iloc[idx[n_train + n_val:]].copy().reset_index(drop=True)
    return train, val, test

def train_linear_regression_reg(X, y, r=0.0):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])
    XTX = X.T.dot(X)
    reg = r * np.eye(XTX.shape[0])
    reg[0, 0] = 0.0
    XTX = XTX + reg
    XTX_inv = np.linalg.inv(XTX)
    w = XTX_inv.dot(X.T).dot(y)
    return w[0], w[1:]

def rmse(y_true, y_pred):
    error = y_pred - y_true
    return np.sqrt((error ** 2).mean())

### Question 3: Filling NAs
Compare filling with 0 vs filling with the mean from the training set.

In [ ]:
df_train, df_val, df_test = split_data(df, seed=42)
y_train = df_train[TARGET].values
y_val = df_val[TARGET].values

# Fill with 0
X_train_0 = df_train[FEATURES].fillna(0).values
X_val_0 = df_val[FEATURES].fillna(0).values
w0_0, w_0 = train_linear_regression_reg(X_train_0, y_train, r=0.0)
rmse_0 = round(rmse(y_val, w0_0 + X_val_0.dot(w_0)), 3)

# Fill with mean
mean_val = df_train['horsepower'].mean()
X_train_mean = df_train[FEATURES].fillna(mean_val).values
X_val_mean = df_val[FEATURES].fillna(mean_val).values
w0_mean, w_mean = train_linear_regression_reg(X_train_mean, y_train, r=0.0)
rmse_mean = round(rmse(y_val, w0_mean + X_val_mean.dot(w_mean)), 3)

print(f'RMSE with 0:    {rmse_0}')
print(f'RMSE with mean: {rmse_mean}')
q3_ans = 'With mean' if rmse_mean < rmse_0 else ('With 0' if rmse_0 < rmse_mean else 'Both are equally good')
print(f'Answer Q3: {q3_ans}')

### Question 4: Best Regularization
Try different values of `r` from `[0, 0.01, 0.1, 1, 5, 10, 100]`. If multiple give the same best RMSE, select the smallest `r`.

In [ ]:
r_candidates = [0, 0.01, 0.1, 1, 5, 10, 100]
r_scores = {}
for r in r_candidates:
    w0, w = train_linear_regression_reg(X_train_0, y_train, r=r)
    score = round(rmse(y_val, w0 + X_val_0.dot(w)), 4)
    r_scores[r] = score
    print(f'r = {r:6} -> Validation RMSE = {score:.4f}')

best_r = min(r_candidates, key=lambda r: (r_scores[r], r))
print(f'\nAnswer Q4: {best_r}')

### Question 5: Seed Stability (Standard Deviation)
Evaluate model performance over seeds `[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]` and compute `np.std(scores)`.

In [ ]:
seed_scores = []
for seed in range(10):
    tr, va, te = split_data(df, seed=seed)
    y_tr = tr[TARGET].values
    y_va = va[TARGET].values
    X_tr = tr[FEATURES].fillna(0).values
    X_va = va[FEATURES].fillna(0).values

    w0, w = train_linear_regression_reg(X_tr, y_tr, r=0.0)
    y_pred = w0 + X_va.dot(w)
    score = rmse(y_va, y_pred)
    seed_scores.append(score)

std_val = round(np.std(seed_scores), 3)
print(f'Scores across seeds: {[round(s, 4) for s in seed_scores]}')
print(f'Standard deviation: {std_val}')
print(f'Answer Q5: {std_val}')

### Question 6: Evaluation on Test Dataset
Split with seed 9, combine train and validation datasets, fill NAs with 0, and train with $r=0.001$.

In [ ]:
tr_9, va_9, te_9 = split_data(df, seed=9)
full_train = pd.concat([tr_9, va_9]).reset_index(drop=True)

y_full = full_train[TARGET].values
y_test = te_9[TARGET].values

X_full = full_train[FEATURES].fillna(0).values
X_test = te_9[FEATURES].fillna(0).values

w0_test, w_test = train_linear_regression_reg(X_full, y_full, r=0.001)
y_pred_test = w0_test + X_test.dot(w_test)
test_rmse = round(rmse(y_test, y_pred_test), 3)
print(f'Test RMSE: {test_rmse}')
print(f'Answer Q6: {test_rmse}')